In [1]:
from typing import cast, Sequence
from sklearn.model_selection import GroupKFold
from sklearn.dummy import DummyRegressor
from pathlib import Path

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from core import Config
import pandas as pd

sector_numbers: Sequence[int] = range(10, 61, 5)
config = Config()

for i in sector_numbers:
    print(f"Training sector {i}")
    DATASET: str = f'imputed_thresh_50_win_log_sector_{i}-r'
    DATA_PATH: Path = config.training_dir / f'{DATASET}.csv'
    # variables
    DEPTH: int = 4
    ITERATIONS: int = 10000
    LEARNING_RATE: float = 0.1
    FOLDS: int = 5
    #ORDERED: str | None = None
    SHUFFLE: bool = True
    STATE: int = 42
    RESULTS_NAME: str = f'Dummy_{DEPTH}_{DATASET}'
    if ORDERED:
        RESULTS_NAME += f'_{ORDERED}'
    if SHUFFLE:
        RESULTS_NAME += f'_sh{STATE}'
    SHAP_NAME: str = f'{RESULTS_NAME}_shap_importance.csv'
    SHAP_PLOT_NAME: str = f'{RESULTS_NAME}_shap_beeswarm.png'

    all_data: pd.DataFrame = pd.read_csv(DATA_PATH, index_col=0)
    training_data = all_data.copy()
    training_data.drop(
        training_data[training_data["TR.UpstreamScope3PurchasedGoodsAndServices"].isna()].index,
        inplace=True
    )
    y: pd.DataFrame = training_data['TR.UpstreamScope3PurchasedGoodsAndServices'].to_frame()
    X: pd.DataFrame = training_data.drop('TR.UpstreamScope3PurchasedGoodsAndServices', axis=1)
    # date could also be a categorical variable
    cat_cols: list[str] = ["Instrument", "TR.GICSSectorCode", "TR.HQCountryCode"]
    cat_cols.remove("Instrument")
    # disable categorical features when using one-hot encoded datasets
    #cat_cols: list[str] = []
    X[cat_cols] = X[cat_cols].astype("str")

    gkf: GroupKFold = GroupKFold(n_splits=FOLDS, random_state=STATE, shuffle=SHUFFLE)
    fold_results: list[dict] = []

    best_fold_index: int | None = None
    best_metric_value: float | None = None  # lower is better (RMSE)
    best_model: DummyRegressor = DummyRegressor()
    best_X_val: pd.DataFrame | None = None
    best_y_val: pd.Series | None = None
    for fold_index, (train_index, val_index) in enumerate(gkf.split(X, y, groups=X['Instrument']), start=1):
        print(f"Fold {fold_index}")
        X_without_instrument = X.drop(columns=["Instrument"])
        X_train: pd.DataFrame = cast(pd.DataFrame, X_without_instrument.iloc[train_index])
        #    X_train: pd.DataFrame = cast(pd.DataFrame, X.iloc[train_index])
        X_val: pd.DataFrame = cast(pd.DataFrame, X_without_instrument.iloc[val_index])
        #    X_val: pd.DataFrame = cast(pd.DataFrame, X.iloc[val_index])
        y_train: pd.Series[float] = cast(pd.Series, y.iloc[train_index])
        y_val: pd.Series[float] = cast(pd.Series, y.iloc[val_index])

        # --- Dummy mean baseline ---
        dummy = DummyRegressor(strategy="mean")
        dummy.fit(X_train, y_train)  # X_train from your existing code

        y_val_pred_dummy = dummy.predict(X_val)

        rmse_dummy = float(np.sqrt(mean_squared_error(y_val, y_val_pred_dummy)))
        mae_dummy = cast(float, mean_absolute_error(y_val, y_val_pred_dummy))
        r2_dummy = r2_score(y_val, y_val_pred_dummy)

        fold_results.append(
            {
                "model_name": "DummyMean",
                "fold": fold_index,
                "sector": "All",
                "rmse": rmse_dummy,
                "mae": mae_dummy,
                "r2": r2_dummy,
            }
        )

        # update best model based on RMSE
        if (best_metric_value is None) or (rmse_dummy < best_metric_value):
            best_metric_value = rmse_dummy
            best_fold_index = fold_index
            best_model = dummy
            best_X_val = X_val.copy()
            best_y_val = y_val.copy()

        val_sectors: pd.Series = X_val["TR.GICSSectorCode"].astype("category")

        val_sectors = X_val["TR.GICSSectorCode"]
        val_df: pd.DataFrame = pd.DataFrame(
            {
                "y_true": y_val.to_numpy().ravel(),
                "y_pred": y_val_pred_dummy,
                "sector": val_sectors.to_numpy().ravel(),
            }
        )

        for sector, grp in val_df.groupby("sector"):
            rmse = float(np.sqrt(mean_squared_error(grp["y_true"], grp["y_pred"])))
            mae = cast(float, mean_absolute_error(grp["y_true"], grp["y_pred"]))
            r2 = r2_score(grp["y_true"], grp["y_pred"])

            fold_results.append(
                {
                    "model_name": 'DummyMean',
                    "fold": fold_index,
                    "sector": sector,
                    "rmse": rmse,
                    "mae": mae,
                    "r2": r2,
                }
            )

    print(f"Best fold by RMSE: {best_fold_index} (RMSE={best_metric_value:.2f})")
    metrics_df: pd.DataFrame = pd.DataFrame(fold_results)

    metrics_df.to_csv(config.results_dir / f'{RESULTS_NAME}_metrics.csv')

Training sector 10
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 1 (RMSE=2.17)
Training sector 15
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 5 (RMSE=1.67)
Training sector 20
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 4 (RMSE=2.31)
Training sector 25
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 3 (RMSE=1.99)
Training sector 30
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 2 (RMSE=2.21)
Training sector 35
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 4 (RMSE=1.91)
Training sector 40
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 3 (RMSE=2.63)
Training sector 45
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 3 (RMSE=2.58)
Training sector 50
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 1 (RMSE=2.33)
Training sector 55
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 2 (RMSE=2.58)
Training sector 60
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 2 (RMSE=2.46)
